In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [28]:
data = pd.read_csv("titanic.csv")
df = pd.DataFrame(data)

In [29]:
# Basic Data set Overview

In [30]:
print("Shape:",df.shape)
print("-"*50)
print("Dtypes:", df.dtypes)
print("-"*50)
print("Info:", df.info)

Shape: (891, 12)
--------------------------------------------------
Dtypes: PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object
--------------------------------------------------
Info: <bound method DataFrame.info of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0        

In [31]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [32]:
df.nunique()

,0
PassengerId,891
Survived,2
Pclass,3
Name,891
Sex,2
Age,88
SibSp,7
Parch,7
Ticket,681
Fare,248


In [33]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [34]:
# Missing value analysis
missng_count = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_count":missng_count,
    "missing_percent":missing_percent
})

print(missing_summary)

             missing_count  missing_percent
Cabin                  687        77.104377
Age                    177        19.865320
Embarked                 2         0.224467
PassengerId              0         0.000000
Name                     0         0.000000
Pclass                   0         0.000000
Survived                 0         0.000000
Sex                      0         0.000000
Parch                    0         0.000000
SibSp                    0         0.000000
Fare                     0         0.000000
Ticket                   0         0.000000


In [35]:
# check for duplicates

duplicates_mask = df.duplicated()
num_duplicates = duplicates_mask.sum()
print("Number of duplicate rows:", num_duplicates)

Number of duplicate rows: 0


In [43]:
n_rows = len(df)
df['constant_cols'] = 1
nunique = df.nunique()
print(nunique)
constant_cols = nunique[nunique == 1].index.to_list()
print(constant_cols)

PassengerId      891
Survived           2
Pclass             3
Name             891
Sex                2
Age               88
SibSp              7
Parch              7
Ticket           681
Fare             248
Cabin            147
Embarked           3
constant_cols      1
dtype: int64
['constant_cols']


In [47]:
# Quasi constant columns
quasi_constant_cols = []
for col in df.columns:
  top_freq = df[col].value_counts(normalize=True, dropna=False).values[0]
  if top_freq > 0.95 and col not in constant_cols:
    quasi_constant_cols.append(col)

  print("Quasi constant columns (top value more than 95 percent): ", quasi_constant_cols)

Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []
Quasi constant columns (top value more than 95 percent):  []


In [51]:
df['Survived'].value_counts(normalize=True).values

array([0.61616162, 0.38383838])

In [52]:
df['id'] = np.arange(1,len(df)+1)

In [56]:
n_rows = len(df)
id_like_cols = []
for col in df.columns:
  if df[col].nunique(dropna=False) == n_rows:
    id_like_cols.append(col)

print("ID like columns:",id_like_cols)

ID like columns: ['PassengerId', 'Name', 'id']


In [64]:
object_cols = df.select_dtypes(include = "object").columns.to_list()
print("Obj cols",object_cols)
df['embark_town_clean'] = (
    df['Embarked'].astype(str)
    .str.strip()
    .str.lower()
    .replace("uknown", np.nan)
  )

Obj cols ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


In [70]:
df.loc[0, "embark_town_clean"]

's'

In [78]:
# high null values
high_null_percentage = 0.4
high_null_cols = missing_summary[missing_summary['missing_percent'] >= high_null_percentage * 100]
print("Columns with high missing percentage:")
print(high_null_cols)

Columns with high missing percentage:
       missing_count  missing_percent
Cabin            687        77.104377


In [87]:
# hig zero columns(0)
numeric_cols = df.select_dtypes(include=[np.number]).columns.to_list()

high_zero_columns = (df[numeric_cols] == 0).mean()
print(high_zero_columns)



PassengerId      0.000000
Survived         0.616162
Pclass           0.000000
Age              0.000000
SibSp            0.682379
Parch            0.760943
Fare             0.016835
constant_cols    0.000000
id               0.000000
dtype: float64


In [90]:
survived = {
    1: "survived",
    0: "not survived"
}

df['target_val'] = df["Survived"].map(survived)
df['target_val']

,target_val
0,not survived
1,survived
2,survived
3,survived
4,not survived
...,...
886,not survived
887,survived
888,not survived
889,survived
